In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
events = pd.read_csv('RetailRocket/events_filtered.csv')
events.head()

,timestamp,visitorid,event,itemid,transactionid,datetime,date
0,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164,2015-06-02
1,1433223291897,794181,view,439202,NaN,2015-06-02 05:34:51.897,2015-06-02
2,1433220899221,824915,view,428805,NaN,2015-06-02 04:54:59.221,2015-06-02
3,1433222531378,57036,view,334662,NaN,2015-06-02 05:22:11.378,2015-06-02
4,1433223203944,125625,view,17655,NaN,2015-06-02 05:33:23.944,2015-06-02


In [3]:
#use only meaningful event strengths 

event_weights = {
    'view': 1,
    'addtocart': 3,
    'transaction': 5
}

events['event_weight'] = events['event'].map(event_weights)

In [4]:
#aggregate by user-item pair

interaction_df = (
    events.groupby(['visitorid', 'itemid'])['event_weight']
    .sum()
    .reset_index()
)

interaction_df.head()

,visitorid,itemid,event_weight
0,2,216305,2
1,2,259884,1
2,2,325215,3
3,2,342816,2
4,6,65273,3


In [5]:
#reduce size before pivoting (keep only top active users and top popular items)

top_users = interaction_df['visitorid'].value_counts().head(3000).index
top_items = interaction_df['itemid'].value_counts().head(1000).index

interaction_subset = interaction_df[
    (interaction_df['visitorid'].isin(top_users)) &
    (interaction_df['itemid'].isin(top_items))
]

interaction_subset.shape

(21803, 3)

In [6]:
#create user-item matrix

user_item_matrix = interaction_subset.pivot(
    index='visitorid',
    columns='itemid',
    values='event_weight'
).fillna(0)

user_item_matrix.shape

(2492, 999)

In [7]:
#create item-item simalarity matrix

#transpose
item_user_matrix = user_item_matrix.T
item_user_matrix.shape

#compute cosine similarity
item_similarity = cosine_similarity(item_user_matrix)
item_similarity.shape

#convert to dataframe
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

item_similarity_df.iloc[:5, :5]

itemid,546,829,869,1152,2836
itemid,,,,,
546,1.000000,0.021725,0.331734,0.363962,0.000000
829,0.021725,1.000000,0.033034,0.029748,0.009879
869,0.331734,0.033034,1.000000,0.397842,0.000000
1152,0.363962,0.029748,0.397842,1.000000,0.018305
2836,0.000000,0.009879,0.000000,0.018305,1.000000


In [8]:
#recommendation function

def get_similar_items(item_id, similarity_df, n=5):
    if item_id not in similarity_df.index:
        return f"Item {item_id} not found in similarity matrix."
    
    similar_scores = similarity_df[item_id].sort_values(ascending=False)
    
    # first result is always the item itself, so skip it
    similar_items = similar_scores.iloc[1:n+1]
    
    return similar_items

In [9]:
#test function
sample_item = item_similarity_df.index[0]
get_similar_items(sample_item, item_similarity_df, n=5)

itemid
248455    0.646174
401285    0.620675
384302    0.595587
332629    0.587948
340825    0.586361
Name: 546, dtype: float64

In [10]:
#format recommendations into dataframe

def recommend_similar_items(item_id, similarity_df, n=5):
    if item_id not in similarity_df.index:
        return pd.DataFrame({'message': [f"Item {item_id} not found."]})
    
    similar_scores = similarity_df[item_id].sort_values(ascending=False).iloc[1:n+1]
    
    recommendations = pd.DataFrame({
        'recommended_itemid': similar_scores.index,
        'similarity_score': similar_scores.values
    })
    
    return recommendations

In [11]:
#test
recommend_similar_items(sample_item, item_similarity_df, n=5)

,recommended_itemid,similarity_score
0,248455,0.646174
1,401285,0.620675
2,384302,0.595587
3,332629,0.587948
4,340825,0.586361


In [12]:
#try few different items

for item in item_similarity_df.index[:3]:
    print(f"\nRecommendations for item {item}:")
    print(recommend_similar_items(item, item_similarity_df, n=5))


Recommendations for item 546:
   recommended_itemid  similarity_score
0              248455          0.646174
1              401285          0.620675
2              384302          0.595587
3              332629          0.587948
4              340825          0.586361

Recommendations for item 829:
   recommended_itemid  similarity_score
0               85579          0.444332
1              298868          0.440600
2               16813          0.431964
3              198209          0.427263
4              369447          0.423905

Recommendations for item 869:
   recommended_itemid  similarity_score
0              272907          0.677166
1              441852          0.588046
2              440066          0.578754
3              130865          0.574801
4              130709          0.544067


In [14]:
#save outputs

item_similarity_df.to_csv('RetailRocket/item_similarity_matrix.csv')
interaction_subset.to_csv('RetailRocket/interaction_subset.csv', index=False)